In [ ]:
import sys
sys.path.append("..")

from src.spark_utils import load_config, get_spark, read_raw_csv, prepare_table, write_to_mysql, read_from_mysql
from src.table_configs import TABLES, LOAD_ORDER, rename_map, type_map
from pyspark.sql import functions as F
from pyspark.sql import Window

In [ ]:
cfg = load_config()
spark = get_spark(cfg["jdbc_jar"])

In [ ]:
EXPECTED_ROWCOUNTS = {
    "district": 77,
    "client": 5369,
    "account": 4500,
    "disp": 5369,
    "card": 892,
    "loan": 682,
    "order": 6471,
    "trans": 1056320,
}

for table_key in LOAD_ORDER:
    table_cfg = TABLES[table_key]
    count_df = read_from_mysql(
        spark, cfg["jdbc_url"], cfg["db_user"], cfg["db_password"],
        dbtable=f"(SELECT COUNT(*) AS cnt FROM {table_cfg['db_table']}) AS t",
    )
    actual = count_df.collect()[0]["cnt"]
    expected = EXPECTED_ROWCOUNTS[table_key]
    status = "OK" if actual == expected else "MISMATCH"
    print(f"{table_key:10s} expected={expected:>8} fact={actual:>8}  [{status}]")

Reading all model tables in one block

In [ ]:
dim_client   = read_from_mysql(spark, cfg["jdbc_url"], cfg["db_user"], cfg["db_password"], "dim_client")
dim_account  = read_from_mysql(spark, cfg["jdbc_url"], cfg["db_user"], cfg["db_password"], "dim_account")
dim_district = read_from_mysql(spark, cfg["jdbc_url"], cfg["db_user"], cfg["db_password"], "dim_district")
dim_date     = read_from_mysql(spark, cfg["jdbc_url"], cfg["db_user"], cfg["db_password"], "dim_date")
bridge       = read_from_mysql(spark, cfg["jdbc_url"], cfg["db_user"], cfg["db_password"], "bridge_client_account")

fact_transactions = read_from_mysql(spark, cfg["jdbc_url"], cfg["db_user"], cfg["db_password"], "fact_transactions")
fact_loans        = read_from_mysql(spark, cfg["jdbc_url"], cfg["db_user"], cfg["db_password"], "fact_loans")
fact_orders       = read_from_mysql(spark, cfg["jdbc_url"], cfg["db_user"], cfg["db_password"], "fact_orders")

results = []

Completeness: referential integrity

In [ ]:
from src.validations import (
    check_referential_integrity, check_uniqueness, check_not_null,
    check_allowed_values, check_running_balance, check_loan_after_account_open,
)

REFERENTIAL_CHECKS = [
    (fact_transactions, "account_id", dim_account,  "account_id",  "fact_transactions -> dim_account"),
    (fact_transactions, "date_key",   dim_date,      "date_key",   "fact_transactions -> dim_date"),
    (fact_loans,         "account_id", dim_account,  "account_id", "fact_loans -> dim_account"),
    (fact_loans,         "date_key",   dim_date,     "date_key",   "fact_loans -> dim_date"),
    (fact_orders,        "account_id", dim_account,  "account_id", "fact_orders -> dim_account"),
    (dim_account,        "district_id", dim_district,"district_id","dim_account -> dim_district"),
    (dim_client,         "district_id", dim_district,"district_id","dim_client -> dim_district"),
    (bridge,             "client_id",  dim_client,   "client_id",  "bridge -> dim_client"),
    (bridge,             "account_id", dim_account,  "account_id", "bridge -> dim_account"),
]

for fact_df, fk_col, dim_df, pk_col, name in REFERENTIAL_CHECKS:
    results.append(check_referential_integrity(fact_df, fk_col, dim_df, pk_col, name))

Uniqueness: checking primary keys

In [ ]:
UNIQUENESS_CHECKS = [
    (dim_client,        ["client_id"],  "dim_client PK"),
    (dim_account,       ["account_id"], "dim_account PK"),
    (fact_transactions, ["trans_id"],   "fact_transactions PK"),
    (fact_loans,        ["loan_id"],    "fact_loans PK"),
    (fact_orders,       ["order_id"],   "fact_orders PK"),
    (bridge, ["client_id", "account_id"], "bridge composite PK"),
]

for df, cols, name in UNIQUENESS_CHECKS:
    results.append(check_uniqueness(df, cols, name))

Completeness (at the column level) і Validity

In [ ]:
results += check_not_null(dim_client, ["birth_date", "gender", "district_id"], "dim_client")
results += check_not_null(fact_transactions, ["amount", "balance", "account_id", "date_key"], "fact_transactions")

DOMAIN_CHECKS = [
    (dim_client,        "gender",           ["M", "F"],                 "dim_client.gender domain"),
    (fact_transactions, "type",             ["PRIJEM", "VYDAJ", "VYBER"],        "fact_transactions.type domain"),
    (fact_loans,        "status",           ["A", "B", "C", "D"],       "fact_loans.status domain"),
    (bridge,            "disposition_type", ["OWNER", "DISPONENT"],     "bridge.disposition_type domain"),
]

for df, col, allowed, name in DOMAIN_CHECKS:
    results.append(check_allowed_values(df, col, allowed, name))

Accuracy і Consistency

In [ ]:
balance_result, balance_mismatches = check_running_balance(fact_transactions)
results.append(balance_result)

results.append(check_loan_after_account_open(fact_loans, dim_account, dim_date))

Final report

In [ ]:
import pandas as pd

report = pd.DataFrame(results)
report

In [ ]:
fact_transactions.groupBy("type").count().show()

  Accuracy check FAIL is expected and investigated in detail in
  `04_investigation_balance_mismatch.ipynb`. Root cause: UROK batch
  ordering + inherent ~0.10 rounding in the source data.